<a href="https://colab.research.google.com/github/Farahani1/kodum/blob/claude%2Fsweet-franklin-pm0tdm/notebooks/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# kodoom on Colab

This notebook only connects Colab to the project: it mounts Drive, gets the code, installs it and runs `kodoom` commands. All logic lives in the repository, so what runs here is exactly what already ran on the laptop (`dev` profile).

**Every session:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

**After a session dies:** reconnect and Run all again. Logs and checkpoints are already on Drive; a run started again with the same run id resumes from its last checkpoint.

**To update the code:** push to the branch from the laptop, then re-run the *Get the code* cell (or Run all). See `docs/colab.md`.

## Settings

In [ ]:
REF = "claude/sweet-franklin-pm0tdm"  # branch, tag or commit to run
PROFILE = "colab-preflight"  # colab-preflight first, then colab
TOKEN_SECRET = "github-kodoom"  # name of the Colab secret holding the GitHub token
HF_TOKEN_SECRET = "HF_TOKEN"  # Colab secret with a Hugging Face read token (gated models)

## Mount Google Drive
Runs, logs and checkpoints are written to `MyDrive/kodoom/runs`. Nothing else goes to Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Get the code
Clones the repository the first time and moves to the latest `REF` every time it runs, discarding any edits made on Colab: code changes are made on the laptop and pushed.

For a private repository, add a GitHub token under 🔑 Secrets (left sidebar) named as `TOKEN_SECRET` in Settings (`github-kodoom`), with notebook access on. The token is used for each fetch and is not saved in the clone.

In [ ]:
import os
import subprocess

REPO = "github.com/Farahani1/kodum.git"
DIR = "/content/kodoom"

token, token_problem = None, None
try:
    from google.colab import userdata

    token = userdata.get(TOKEN_SECRET)
except Exception as e:  # a public repository needs no token
    token_problem = f"{type(e).__name__}: {e}"
url = f"https://x-access-token:{token}@{REPO}" if token else f"https://{REPO}"


def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        if not token and "Username" in message:
            message += (
                "\nThe repository is private and no token was used "
                f"({token_problem or TOKEN_SECRET + ' is empty'}). In Secrets (key icon, "
                f"left sidebar): name it exactly {TOKEN_SECRET}, paste the github_pat_... "
                "value, turn Notebook access on, then re-run this cell."
            )
        raise RuntimeError(message)  # git's own message, never the token
    return result.stdout.strip()


if not os.path.isdir(f"{DIR}/.git"):
    git("init", "--quiet", DIR)
git("-C", DIR, "fetch", "--quiet", "--depth", "50", url, REF)
git("-C", DIR, "checkout", "--quiet", "--force", "FETCH_HEAD")
print(git("-C", DIR, "log", "-1", "--format=%h %s (%cr)"))

In [ ]:
!pip install --quiet -e "{DIR}[colab]"

## Check the environment
Stops here if Drive is not mounted, no GPU is attached or Drive is too full. Also lists the runs that can be resumed.

In [ ]:
!kodoom check --profile {PROFILE}

In [ ]:
!kodoom runs --profile {PROFILE}

## Record the data before this run
Remembers which files are in `data_dir` now, so the last cell can name the files this run generates or updates. It also brings `README.md` in that folder up to date.

In [ ]:
!kodoom tree --profile {PROFILE} --start

## Run
Pipeline steps are added here as they are built, each as one `kodoom` command with `--profile {PROFILE}`. Re-running a step with the same run id resumes it.

## Data

The skills data is generated, not downloaded: same seed, same files, in seconds. It is written to **your Drive** (`MyDrive/kodoom/data/skills`, see `data_dir` in the profile), never into the repository checkout, and nothing here pushes data anywhere. A manifest next to each file holds the seed and a checksum.

In [ ]:
!kodoom generate --profile {PROFILE}

### Review pack for a native reader

Writes the glossary and every Persian template, each next to an item generated from it, to **your Drive** (`.../review/`): `glossary.html` and `templates.html` to read in a browser, and `glossary.csv` and `templates.csv` with empty `ok`, `suggestion` and `note` columns to fill. Save filled sheets under a new name (e.g. `templates-reviewed.csv`): this cell rewrites the originals.

In [ ]:
!kodoom review-pack --profile {PROFILE}

## Hugging Face access (optional, needed for TranslateGemma)

Some models are gated. Once: open the model page, accept the license, create a **read** token at huggingface.co/settings/tokens, and add it under 🔑 Secrets as `HF_TOKEN` with notebook access on. This cell passes it to the commands below; without a secret everything else still works (with slower downloads).

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get(HF_TOKEN_SECRET)
    print("Hugging Face token set")
except Exception as e:  # not needed unless a model is gated
    print(f"no Hugging Face token ({type(e).__name__}); gated models will fail")

## Fetch typed-decisions (English originals)

Downloads `LocalLLaMA/typed-decisions` at its pinned commit and converts it to records on **your Drive** (`MyDrive/kodoom/data/.../typed-decisions/en`). Nothing is pushed anywhere. Preflight takes the first 200 cases per workflow and split; the `colab` profile takes all. **Copy the output of both cells back to the assistant.**

In [ ]:
!kodoom fetch typed-decisions --profile {PROFILE}

In [ ]:
!kodoom fields --profile {PROFILE}

## Fetch helmo/synthetic-typed-decisions (English)

The extra topic-breadth data (plan 1.1), converted to records on **your Drive** (`.../helmo/en`). Preflight takes the first 200 records; the `colab` profile takes all 9,879. **Copy the output back to the assistant.**

In [ ]:
!kodoom fetch helmo --profile {PROFILE}

## Translation trial (10 cases)

Translates 10 test cases and prints the automatic checks and English next to Persian. Results so far: TranslateGemma 4B (bf16, the only setting that works on the T4) ignores the glossary; writing the terms into its input did not hold; instruction-tuned Gemma 3 with the glossary in the prompt did best. The first cells are kept for comparison and skip cases already done. Output goes to **your Drive** (`.../typed-decisions/fa/<translator>`); nothing is pushed. **Copy the output of each cell back to the assistant**, including errors.

In [ ]:
T = "translategemma-4b-bf16"
!kodoom translate typed-decisions --profile {PROFILE} --translator {T} --split test --limit 10
!kodoom translations typed-decisions --profile {PROFILE} --translator {T} --show 3

Same model with the glossary terms written into the input in Persian, so that «agent», «trace» and the like are translated the same way everywhere. Compare its *findings by check* with the plain run above (which had 132 glossary findings).

In [ ]:
T = "translategemma-4b-bf16-terms"
!kodoom translate typed-decisions --profile {PROFILE} --translator {T} --split test --limit 10
!kodoom translations typed-decisions --profile {PROFILE} --translator {T} --show 3

### Instruction-tuned Gemma 3 (glossary and context in the prompt)

TranslateGemma cannot take instructions, and writing the glossary into its input did not hold (132 → 110 findings; it turns «agent» back into «person»). `gemma3-4b-bf16` is a general chat model: its prompt carries the glossary, the register and a sentence on where the text comes from. **Needs access to `google/gemma-3-4b-it`:** accept its license on Hugging Face with the same account (separate from TranslateGemma's).

In [ ]:
T = "gemma3-4b-bf16"
!kodoom translate typed-decisions --profile {PROFILE} --translator {T} --split test --limit 10
!kodoom translations typed-decisions --profile {PROFILE} --translator {T} --show 3

### Free disk space before the 12B model

The Colab disk was at 79 of 112.6 GB after the earlier models, and the 12B weights are about 24 GB. This removes the Qwen3-8B download (16 GB, no longer needed) and lists what is left in the model cache. It never touches Drive.

In [ ]:
!rm -rf /content/hf-cache/hub/models--Qwen--Qwen3-8B
!du -sh /content/hf-cache/hub/models--* 2>/dev/null
!df -h /content | tail -1

The 12B version in 4-bit (about 24 GB to download, 7–8 GB in memory; needs access to `google/gemma-3-12b-it`). Run it after the 4B one looks promising.

In [ ]:
T = "gemma3-12b-4bit"
!kodoom translate typed-decisions --profile {PROFILE} --translator {T} --split test --limit 10
!kodoom translations typed-decisions --profile {PROFILE} --translator {T} --show 3

## Pilot: 50 cases, two translators, blind review (plan 1.2 step 1)

Both translators translate the **same 50 training cases**, taken evenly from the four workflows (training cases, so the test cases are not reviewed twice). Then one command writes a spreadsheet in which each case appears once with the two translations in random order as A and B. **You read it, without knowing which is which**, and fill in `better` (A, B or tie) and the number of meaning errors per side. The answer key is written to a separate file you should not open until you are done. Output is on **your Drive** in `MyDrive/kodoom/data/preflight/typed-decisions/pilot/`.

In [ ]:
T = "gemma3-4b-bf16"
PILOT = "--split train --limit 50 --balanced"
!kodoom translate typed-decisions --profile {PROFILE} --translator {T} {PILOT}

In [ ]:
T = "gemma3-12b-4bit"
PILOT = "--split train --limit 50 --balanced"
!kodoom translate typed-decisions --profile {PROFILE} --translator {T} {PILOT}

In [ ]:
A, B = "gemma3-4b-bf16", "gemma3-12b-4bit"
!kodoom pilot-sheet typed-decisions --profile {PROFILE} --a {A} --b {B}

After filling the sheet and saving it as `sheet-filled.csv` in the same `pilot/` folder, tally it against the key:

In [ ]:
!kodoom pilot-score typed-decisions --profile {PROFILE}

## Translating with Claude Cowork or a person (optional)

Exports each distinct English text once, with its register, context and glossary terms, plus a brief for the translator. After the file is filled, the import step builds the Persian records and runs all the automatic checks. Steps and file names are in `docs/colab.md`. The 50 balanced training cases of the pilot are a good first batch.

In [ ]:
OPTS = "--split train --limit 50 --balanced"
!kodoom export-units typed-decisions --profile {PROFILE} {OPTS}

In [ ]:
!kodoom import-units typed-decisions --profile {PROFILE} --name cowork {OPTS}

In [ ]:
!kodoom info --profile {PROFILE}

## Files generated or updated in this run

The tree of everything in `data_dir` on **your Drive**, with sizes. Files this run made are marked `[new]`, files it changed `[updated]`, and the list under the tree names them again (compared with the files recorded by the *Record the data* cell at the start). The same tree, with record counts and the history of every command that wrote data, is in `README.md` in that folder; each data command keeps it up to date.

In [ ]:
!kodoom tree --profile {PROFILE}

## Before closing (optional)
Everything is already on Drive as it is written. This forces Drive to finish syncing, for peace of mind before closing the tab.

In [ ]:
drive.flush_and_unmount()